# 스트레스 점수 예측 V24 — ExtraTrees Structural Audit Search

- Public 0.1272333 V7의 파생변수·Pair-Neighbor 15%는 고정
- ExtraTrees의 `max_features`, `min_samples_leaf`, `criterion`, `bootstrap`을 구조적으로 탐색
- Development CV(seed 42)에서 200 Tree로 48개 구조를 1차 선별
- 상위 5개만 사전에 고정하고 Audit CV(seed 77, 2026)에서 1,200 Tree로 재검증
- Audit 두 분할 모두 원본 V7을 이기고 평균 0.0005 이상 개선될 때만 제출 후보
- 모든 전처리·Pair 순위는 각 Fold Train으로만 학습

> 모델 학습량이 많아 실행 시간이 오래 걸립니다.

In [19]:
from __future__ import annotations

import argparse
import os
from itertools import combinations
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold


TARGET = "stress_score"
ID_COLUMN = "ID"
RANDOM_SEED = 42
N_ESTIMATORS = 1200
TREE_QUANTILE = 0.52
PAIR_QUANTILE = 0.52
PAIR_WEIGHT = 0.15

FEATURE_COPY_COUNTS = {
    "mean_working": 1,
    "bmi": 4,
    "cholesterol": 2,
    "height": 2,
    "glucose": 3,
    "weight": 2,
    "cholesterol_glucose_ratio": 2,
    "bone_density": 4,
}

PAIR_COLUMNS = [
    "mean_working",
    "bmi",
    "cholesterol",
    "height",
    "glucose",
    "weight",
    "cholesterol_glucose_ratio",
    "bone_density",
]


def add_features(frame: pd.DataFrame) -> pd.DataFrame:
    """다른 행이나 Test 전체 통계를 사용하지 않는 행 단위 파생변수."""
    result = frame.copy()
    height_m = result["height"] / 100.0
    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]
    result["bmi"] = result["weight"] / height_m.pow(2)
    result["pulse_pressure"] = sbp - dbp
    result["mean_arterial_pressure"] = (sbp + 2.0 * dbp) / 3.0
    result["blood_pressure_ratio"] = sbp / (dbp + 1e-6)
    result["cholesterol_glucose_ratio"] = result["cholesterol"] / (
        result["glucose"] + 1e-6
    )
    result["missing_count"] = result.isna().sum(axis=1)
    return result


def add_feature_copies(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    for feature, additional_copies in FEATURE_COPY_COUNTS.items():
        for copy_index in range(1, additional_copies + 1):
            result[f"{feature}__copy{copy_index}"] = result[feature]
    return result


def make_preprocessor(frame: pd.DataFrame) -> ColumnTransformer:
    categorical = frame.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()
    numerical = frame.select_dtypes(include=["number"]).columns.tolist()
    return ColumnTransformer(
        [
            (
                "categorical",
                Pipeline(
                    [
                        (
                            "imputer",
                            SimpleImputer(
                                strategy="constant", fill_value="missing"
                            ),
                        ),
                        (
                            "encoder",
                            OneHotEncoder(
                                handle_unknown="ignore", sparse_output=False
                            ),
                        ),
                    ]
                ),
                categorical,
            ),
            (
                "numerical",
                SimpleImputer(strategy="median", add_indicator=True),
                numerical,
            ),
        ]
    )


def tree_quantile_prediction(
    train_features: pd.DataFrame,
    test_features: pd.DataFrame,
    target: pd.Series,
) -> np.ndarray:
    weighted_train = add_feature_copies(train_features)
    weighted_test = add_feature_copies(test_features)
    pipeline = Pipeline(
        [
            ("preprocessor", make_preprocessor(weighted_train)),
            (
                "model",
                ExtraTreesRegressor(
                    n_estimators=N_ESTIMATORS,
                    min_samples_leaf=1,
                    max_features=1,
                    random_state=RANDOM_SEED,
                    n_jobs=1,
                ),
            ),
        ]
    )
    pipeline.fit(weighted_train, target)
    matrix = pipeline.named_steps["preprocessor"].transform(weighted_test)
    forest = pipeline.named_steps["model"]
    tree_predictions = np.column_stack(
        [tree.predict(matrix) for tree in forest.estimators_]
    )
    return np.clip(
        np.round(np.quantile(tree_predictions, TREE_QUANTILE, axis=1), 2),
        0.0,
        1.0,
    )


def empirical_rank_transform(
    train_values: np.ndarray,
    test_values: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Train 분포만으로 Train과 한 개씩 독립적인 Test 값을 변환."""
    median = np.nanmedian(train_values)
    train_values = np.nan_to_num(train_values, nan=median)
    test_values = np.nan_to_num(test_values, nan=median)
    sorted_train = np.sort(train_values)
    train_rank = np.searchsorted(
        sorted_train, train_values, side="right"
    ) / len(train_values)
    test_rank = np.searchsorted(
        sorted_train, test_values, side="right"
    ) / len(train_values)
    return train_rank.astype(np.float32), test_rank.astype(np.float32)


def pair_neighbor_prediction(
    train_features: pd.DataFrame,
    test_features: pd.DataFrame,
    target: pd.Series,
) -> np.ndarray:
    """각 2차원 피처쌍에서 가장 가까운 Train target을 분위수 집계."""
    train_rank = {}
    test_rank = {}
    for column in PAIR_COLUMNS:
        train_rank[column], test_rank[column] = empirical_rank_transform(
            train_features[column].to_numpy(dtype=float),
            test_features[column].to_numpy(dtype=float),
        )

    target_values = target.to_numpy(dtype=float)
    neighbor_targets = []
    for first, second in combinations(PAIR_COLUMNS, 2):
        distance = (
            np.abs(test_rank[first][:, None] - train_rank[first][None, :])
            + np.abs(test_rank[second][:, None] - train_rank[second][None, :])
        )
        nearest_index = np.argmin(distance, axis=1)
        neighbor_targets.append(target_values[nearest_index])
    return np.quantile(
        np.column_stack(neighbor_targets), PAIR_QUANTILE, axis=1
    )


def validate_inputs(
    train: pd.DataFrame,
    test: pd.DataFrame,
    submission: pd.DataFrame,
) -> None:
    if set(train.columns) - {TARGET} != set(test.columns):
        raise ValueError("Train과 Test의 입력 컬럼이 다릅니다.")
    if list(submission.columns) != [ID_COLUMN, TARGET]:
        raise ValueError("제출 양식 컬럼이 올바르지 않습니다.")
    if not submission[ID_COLUMN].equals(test[ID_COLUMN]):
        raise ValueError("제출 양식과 Test의 ID 순서가 다릅니다.")
    if train[TARGET].isna().any():
        raise ValueError("Train 타깃에 결측값이 있습니다.")

## 1. 설정과 데이터

In [20]:
DATA_DIR=Path("/Users/bitsaem/Desktop/stress_project/open")
OUTPUT_DIR=Path("/Users/bitsaem/Desktop/stress_project")
DEVELOPMENT_SEED=42
AUDIT_SEEDS=(77,2026)
SCREEN_TREES=200
AUDIT_TREES=1200
TOP_K=5
QUANTILES=(0.50,0.52,0.54)

MAX_FEATURES=(1,2,4,"sqrt")
MIN_LEAVES=(1,2,3)
CRITERIA=("squared_error","absolute_error")
BOOTSTRAPS=(False,True)

train=pd.read_csv(DATA_DIR/"train.csv")
test=pd.read_csv(DATA_DIR/"test.csv")
submission=pd.read_csv(DATA_DIR/"sample_submission.csv")
validate_inputs(train,test,submission)
X=add_features(train.drop(columns=[TARGET,ID_COLUMN])).reset_index(drop=True)
X_test=add_features(test.drop(columns=[ID_COLUMN])).reset_index(drop=True)
y=train[TARGET].reset_index(drop=True)
print("Train:",X.shape,"| structural configs:",
      len(MAX_FEATURES)*len(MIN_LEAVES)*len(CRITERIA)*len(BOOTSTRAPS))

Train: (3000, 22) | structural configs: 48


## 2. 학습 보조 함수

In [21]:
def transform_fold(train_frame,valid_frame):
    weighted_train=add_feature_copies(train_frame)
    weighted_valid=add_feature_copies(valid_frame)
    preprocessor=make_preprocessor(weighted_train)
    train_matrix=preprocessor.fit_transform(weighted_train)
    valid_matrix=preprocessor.transform(weighted_valid)
    return train_matrix,valid_matrix

def fit_tree_material(train_matrix,valid_matrix,target,config,n_estimators):
    max_features,min_leaf,criterion,bootstrap=config
    model=ExtraTreesRegressor(
        n_estimators=n_estimators,min_samples_leaf=min_leaf,
        max_features=max_features,criterion=criterion,
        bootstrap=bootstrap,max_samples=.8 if bootstrap else None,
        random_state=RANDOM_SEED,n_jobs=-1,
    )
    model.fit(train_matrix,target)
    return np.column_stack([
        tree.predict(valid_matrix) for tree in model.estimators_
    ])

def tree_q(material,quantile):
    return np.clip(np.round(np.quantile(material,quantile,axis=1),2),0,1)

def final_blend(tree_prediction,pair_prediction):
    return np.clip(np.round(.85*tree_prediction+.15*pair_prediction,2),0,1)

CONFIGS=[
    (mf,leaf,criterion,bootstrap)
    for mf in MAX_FEATURES for leaf in MIN_LEAVES
    for criterion in CRITERIA for bootstrap in BOOTSTRAPS
]
BASE_CONFIG=(1,1,"squared_error",False)

## 3. Development CV — 48개 구조 1차 선별

In [22]:
cv=KFold(5,shuffle=True,random_state=DEVELOPMENT_SEED)
dev_oof={(config,q):np.zeros(len(y)) for config in CONFIGS for q in QUANTILES}
dev_pair=np.zeros(len(y))

for fold,(ti,vi) in enumerate(cv.split(X),1):
    train_matrix,valid_matrix=transform_fold(X.iloc[ti],X.iloc[vi])
    dev_pair[vi]=pair_neighbor_prediction(X.iloc[ti],X.iloc[vi],y.iloc[ti])
    for number,config in enumerate(CONFIGS,1):
        material=fit_tree_material(
            train_matrix,valid_matrix,y.iloc[ti],config,SCREEN_TREES
        )
        for quantile in QUANTILES:
            dev_oof[(config,quantile)][vi]=tree_q(material,quantile)
        if number%8==0:
            print(f"Fold {fold} | {number}/{len(CONFIGS)} configs")
    print(f"Development Fold {fold}: complete")

screen_base=final_blend(dev_oof[(BASE_CONFIG,.52)],dev_pair)
screen_base_mae=mean_absolute_error(y,screen_base)
screen_rows=[]
for config in CONFIGS:
    for quantile in QUANTILES:
        pred=final_blend(dev_oof[(config,quantile)],dev_pair)
        score=mean_absolute_error(y,pred)
        screen_rows.append({
            "config":config,"quantile":quantile,"development_MAE":score,
            "development_improvement":screen_base_mae-score,
        })
screen_results=pd.DataFrame(screen_rows).sort_values(
    ["development_MAE"]
).reset_index(drop=True)
top_candidates=[]
for _,row in screen_results.iterrows():
    candidate=(row["config"],float(row["quantile"]))
    if candidate not in top_candidates:
        top_candidates.append(candidate)
    if len(top_candidates)==TOP_K:break

print("200-tree development baseline MAE:",f"{screen_base_mae:.6f}")
print("Frozen top candidates:")
for candidate in top_candidates:print(candidate)
screen_results.head(15)

Fold 1 | 8/48 configs
Fold 1 | 16/48 configs
Fold 1 | 24/48 configs
Fold 1 | 32/48 configs
Fold 1 | 40/48 configs
Fold 1 | 48/48 configs
Development Fold 1: complete
Fold 2 | 8/48 configs
Fold 2 | 16/48 configs
Fold 2 | 24/48 configs
Fold 2 | 32/48 configs
Fold 2 | 40/48 configs
Fold 2 | 48/48 configs
Development Fold 2: complete
Fold 3 | 8/48 configs
Fold 3 | 16/48 configs
Fold 3 | 24/48 configs
Fold 3 | 32/48 configs
Fold 3 | 40/48 configs
Fold 3 | 48/48 configs
Development Fold 3: complete
Fold 4 | 8/48 configs
Fold 4 | 16/48 configs
Fold 4 | 24/48 configs
Fold 4 | 32/48 configs
Fold 4 | 40/48 configs
Fold 4 | 48/48 configs
Development Fold 4: complete
Fold 5 | 8/48 configs
Fold 5 | 16/48 configs
Fold 5 | 24/48 configs
Fold 5 | 32/48 configs
Fold 5 | 40/48 configs
Fold 5 | 48/48 configs
Development Fold 5: complete
200-tree development baseline MAE: 0.149507
Frozen top candidates:
((2, 1, 'absolute_error', False), 0.5)
((2, 1, 'absolute_error', False), 0.52)
((1, 1, 'absolute_error'

,config,quantile,development_MAE,development_improvement
0,"(2, 1, absolute_error, False)",0.50,0.147900,0.001607
1,"(2, 1, absolute_error, False)",0.52,0.147967,0.001540
2,"(1, 1, absolute_error, False)",0.52,0.148020,0.001487
3,"(1, 1, absolute_error, False)",0.50,0.148097,0.001410
4,"(1, 1, absolute_error, False)",0.54,0.148500,0.001007
5,"(2, 1, absolute_error, False)",0.54,0.148590,0.000917
6,"(4, 1, absolute_error, False)",0.52,0.148930,0.000577
7,"(4, 1, absolute_error, False)",0.54,0.148983,0.000523
8,"(2, 1, squared_error, False)",0.52,0.149100,0.000407
9,"(4, 1, absolute_error, False)",0.50,0.149143,0.000363


## 4. Audit CV — 고정된 상위 5개를 1,200 Tree로 검증

In [23]:
audit_records=[]
audit_score_rows=[]
unique_configs=list(dict.fromkeys([BASE_CONFIG]+[c for c,q in top_candidates]))

for audit_seed in AUDIT_SEEDS:
    cv=KFold(5,shuffle=True,random_state=audit_seed)
    pair_oof=np.zeros(len(y))
    tree_oof={(config,q):np.zeros(len(y)) for config,q in top_candidates}
    base_oof=np.zeros(len(y))
    for fold,(ti,vi) in enumerate(cv.split(X),1):
        train_matrix,valid_matrix=transform_fold(X.iloc[ti],X.iloc[vi])
        pair_oof[vi]=pair_neighbor_prediction(X.iloc[ti],X.iloc[vi],y.iloc[ti])
        materials={}
        for config in unique_configs:
            materials[config]=fit_tree_material(
                train_matrix,valid_matrix,y.iloc[ti],config,AUDIT_TREES
            )
        base_oof[vi]=tree_q(materials[BASE_CONFIG],.52)
        for config,quantile in top_candidates:
            tree_oof[(config,quantile)][vi]=tree_q(materials[config],quantile)
        print(f"Audit {audit_seed} | Fold {fold}: complete")

    baseline=final_blend(base_oof,pair_oof)
    baseline_mae=mean_absolute_error(y,baseline)
    for config,quantile in top_candidates:
        prediction=final_blend(tree_oof[(config,quantile)],pair_oof)
        score=mean_absolute_error(y,prediction)
        audit_records.append({
            "audit_seed":audit_seed,"config":config,"quantile":quantile,
            "baseline_MAE":baseline_mae,"candidate_MAE":score,
            "improvement":baseline_mae-score,
        })
    print(f"Audit {audit_seed} baseline MAE: {baseline_mae:.6f}")



Audit 77 | Fold 1: complete
Audit 77 | Fold 2: complete
Audit 77 | Fold 3: complete
Audit 77 | Fold 4: complete
Audit 77 | Fold 5: complete
Audit 77 baseline MAE: 0.146387
Audit 2026 | Fold 1: complete
Audit 2026 | Fold 2: complete
Audit 2026 | Fold 3: complete
Audit 2026 | Fold 4: complete
Audit 2026 | Fold 5: complete
Audit 2026 baseline MAE: 0.144733


In [33]:
audit_frame = pd.DataFrame(audit_records)
audit_score_rows = []

for config, quantile in top_candidates:
    mask = (
        audit_frame["config"].map(
            lambda value: tuple(value) == tuple(config)
        )
        & np.isclose(
            audit_frame["quantile"].astype(float),
            float(quantile),
        )
    )
    part = audit_frame[mask]

    audit_score_rows.append({
        "config": tuple(config),
        "quantile": float(quantile),
        "audit_mean_MAE": part["candidate_MAE"].mean(),
        "audit_mean_improvement": part["improvement"].mean(),
        "audit_worst_improvement": part["improvement"].min(),
        "audit_wins": int((part["improvement"] > 0).sum()),
    })

audit_results = pd.DataFrame(audit_score_rows).sort_values(
    ["audit_wins", "audit_worst_improvement", "audit_mean_MAE"],
    ascending=[False, False, True],
).reset_index(drop=True)

audit_results

,config,quantile,audit_mean_MAE,audit_mean_improvement,audit_worst_improvement,audit_wins
0,"(1, 1, absolute_error, False)",0.52,0.145172,0.000388,0.000357,2
1,"(1, 1, absolute_error, False)",0.54,0.145202,0.000358,0.000210,2
2,"(2, 1, absolute_error, False)",0.52,0.145342,0.000218,0.000203,2
3,"(1, 1, absolute_error, False)",0.50,0.145303,0.000257,0.000130,2
4,"(2, 1, absolute_error, False)",0.50,0.145548,0.000012,-0.000167,1


## 5. 최종 후보 결정

In [ ]:
eligible=audit_results[
    (audit_results.audit_wins==len(AUDIT_SEEDS))
    &(audit_results.audit_worst_improvement>0)
    &(audit_results.audit_mean_improvement>=.0003)
]
if len(eligible):
    selected=eligible.iloc[0];selected_config=selected.config
    selected_quantile=float(selected.quantile);use_new_model=True
else:
    selected_config=BASE_CONFIG;selected_quantile=.52;use_new_model=False
    selected=pd.Series({
        "audit_mean_MAE":audit_frame.baseline_MAE.groupby(
            audit_frame.audit_seed
        ).first().mean(),
        "audit_mean_improvement":0.0,"audit_worst_improvement":0.0,
        "audit_wins":0,
    })

print("Selected config:",selected_config)
print("Selected quantile:",selected_quantile)
print("Audit mean MAE:",f"{selected.audit_mean_MAE:.6f}")
print("Audit mean improvement:",f"{selected.audit_mean_improvement:+.6f}")
print("Audit worst improvement:",f"{selected.audit_worst_improvement:+.6f}")
print("Audit wins:",int(selected.audit_wins),"/",len(AUDIT_SEEDS))
print("New model accepted:",use_new_model)

Selected config: (1, 1, 'absolute_error', False)
Selected quantile: 0.52
Audit mean MAE: 0.145172
Audit mean improvement: +0.000388
Audit worst improvement: +0.000357
Audit wins: 2 / 2
New model accepted: False


## 6. 전체 Train 학습과 Test 예측

In [26]:
train_matrix,test_matrix=transform_fold(X,X_test)
material=fit_tree_material(
    train_matrix,test_matrix,y,selected_config,AUDIT_TREES
)
tree_prediction=tree_q(material,selected_quantile)
pair_prediction=pair_neighbor_prediction(X,X_test,y)
pred_v24=final_blend(tree_prediction,pair_prediction)
print("Prediction range:",pred_v24.min(),pred_v24.max())
print("첫 10개:",pred_v24[:10])

Prediction range: 0.0 1.0
첫 10개: [0.56 0.97 0.19 0.48 0.53 0.47 0.78 0.48 0.57 0.88]


## 7. 제출 파일

In [27]:
submission_v24=submission.copy();submission_v24[TARGET]=pred_v24
output_path=OUTPUT_DIR/"submit_v24_structural_audit_search.csv"
submission_v24.to_csv(output_path,index=False,encoding="utf-8")
print("Saved:",output_path)
submission_v24.head(10)

Saved: /Users/bitsaem/Desktop/stress_project/submit_v24_structural_audit_search.csv


,ID,stress_score
0,TEST_0000,0.56
1,TEST_0001,0.97
2,TEST_0002,0.19
3,TEST_0003,0.48
4,TEST_0004,0.53
5,TEST_0005,0.47
6,TEST_0006,0.78
7,TEST_0007,0.48
8,TEST_0008,0.57
9,TEST_0009,0.88


## 제출 기준

- `New model accepted: True`
- `Audit wins: 2 / 2`
- `Audit mean improvement >= 0.0005`
- `Audit worst improvement > 0`
- 조건 미달이면 Public 0.1272333 원본 V7 유지